# BRCA1 PLM Layer DeltaEmbSAE Analysis

This notebook runs the fixed best-AUC sparse autoencoder setting across every layer of each available sub-1B PLM, then compares inferred fitness to MaveDB functional scores and ClinVar benign/pathogenic classifications. The fixed SAE setting is `DeltaEmbSAE`, `max_pool`, `n_features=12800`, `sparsity_mode="batchtopk"`, `k=64`, matching the best BRCA1 AUC configuration from `real_data_esm_analysis_cpu_sae.ipynb`.

Large PLMs over 1B parameters are kept in the model registry but are excluded from default submission so custom GPU jobs can be handled separately.


In [ ]:
from pathlib import Path
import json
import os
import pickle
import re
import subprocess
import sys
from typing import get_args

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from esmDMS import CellularDMSInput, ESMDMSConfig, EmbeddingModel, esmDMS

sns.set_theme(style="darkgrid")
REPO_ROOT


## Configure Data And Fixed SAE

All new artifacts go under `data/esm_data_analysis/BRCA1_plm_layer_sae`. Cached ClinVar annotations and exact popDMS baseline tables are reused from the existing BRCA1 analysis directory when present.


In [ ]:
DATA_DIR = REPO_ROOT / "data" / "mavedb_data"
ANALYSIS_DIR = REPO_ROOT / "data" / "esm_data_analysis" / "BRCA1_plm_layer_sae"
BASELINE_ANALYSIS_DIR = REPO_ROOT / "data" / "esm_data_analysis" / "BRCA1_experimental"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"
JOB_DIR = SEQUENCE_DIR / "jobs"

BASELINE_TABLE_DIR = BASELINE_ANALYSIS_DIR / "tables"
BASELINE_SEQUENCE_DIR = BASELINE_ANALYSIS_DIR / "sequence_data"

for directory in (SEQUENCE_DIR, FIGURE_DIR, TABLE_DIR, JOB_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BRCA1_INPUT = CellularDMSInput(
    reference_nuc_path=DATA_DIR / "BRCA1_reference_sequence.dat",
    mavedb_csv_path=DATA_DIR / "BRCA1_counts.csv",
    scores_csv_path=DATA_DIR / "BRCA1_scores.csv",
    reference_kind="protein",
    primary_key="hgvs_nt",
)

DATASET_NAME = "BRCA1"
EMBEDDING_TYPE = "max_pool"
ABSTRACTION_METHOD = "DeltaEmbSAE"
NORM_SCHEME = "none"
FIGURE_DPI = 450

BEST_SAE_PARAMS = {
    "n_features": 12800,
    "sparsity_coeff": 1e-3,
    "sparsity_mode": "batchtopk",
    "k": 64,
    "epochs": 200,
    "batch_size": 64,
    "train_frac": 0.8,
    "lr": 1e-3,
    "seed": 42,
    "norm_scheme": NORM_SCHEME,
    "run_label": "DeltaEmbSAE_max_pool_batchtopk_k64_nf12800_seed42",
}

CLINVAR_ANNOTATION_CACHE_PATH = BASELINE_TABLE_DIR / "BRCA1_clinvar_annotations_by_hgvs.csv"
EXACT_POPDMS_FITNESS_PATH = BASELINE_TABLE_DIR / "BRCA1_exact_popDMS_fitness_values.csv"
EXACT_POPDMS_SELECTION_PATH = BASELINE_SEQUENCE_DIR / "regular_popdms" / "BRCA1_exact_popDMS_selection_coefficients.csv.gz"
ENRICHMENT_RATIO_FITNESS_PATH = TABLE_DIR / "BRCA1_enrichment_ratio_fitness_values.csv"

SPEARMAN_AUC_TABLE_PATH = TABLE_DIR / "BRCA1_plm_layer_mavedb_spearman_vs_clinvar_auc.csv"
SPEARMAN_AUC_FIGURE_PATH = FIGURE_DIR / "BRCA1_plm_layer_mavedb_spearman_vs_clinvar_auc.png"
SPEARMAN_AUC_STAR_TABLE_TEMPLATE = "BRCA1_plm_layer_mavedb_spearman_vs_clinvar_auc_min_{stars}_stars.csv"
SPEARMAN_AUC_STAR_FIGURE_TEMPLATE = "BRCA1_plm_layer_mavedb_spearman_vs_clinvar_auc_min_{stars}_stars.png"
SPEARMAN_AUC_STAR_COMBINED_TABLE_PATH = TABLE_DIR / "BRCA1_plm_layer_mavedb_spearman_vs_clinvar_auc_by_review_stars.csv"
CLINVAR_REVIEW_STAR_THRESHOLDS = [1, 2, 3, 4]

BEST_SAE_PARAMS


## PLM Registry

`ACTIVE_MODELS` excludes models over 1B parameters by default. Toggle `INCLUDE_CUSTOM_GPU_MODELS_IN_ANALYSIS` after custom large-model embeddings and SAE outputs exist.


In [ ]:
AVAILABLE_MODEL_NAMES = list(get_args(EmbeddingModel))

MODEL_SIZE_MILLIONS = {
    "facebook/esm2_t6_8M_UR50D": 8,
    "facebook/esm2_t12_35M_UR50D": 35,
    "facebook/esm2_t30_150M_UR50D": 150,
    "facebook/esm2_t33_650M_UR50D": 650,
    "facebook/esm2_t36_3B_UR50D": 3000,
    "biohub/ESMC-300M": 300,
    "biohub/ESMC-600M": 600,
    "biohub/ESMC-6B": 6000,
}
MODEL_LAYER_COUNTS = {
    "biohub/ESMC-300M": 30,
    "biohub/ESMC-600M": 36,
    "biohub/ESMC-6B": 80,
}
MODEL_SHORT_NAMES = {
    "facebook/esm2_t6_8M_UR50D": "ESM2-8M",
    "facebook/esm2_t12_35M_UR50D": "ESM2-35M",
    "facebook/esm2_t30_150M_UR50D": "ESM2-150M",
    "facebook/esm2_t33_650M_UR50D": "ESM2-650M",
    "facebook/esm2_t36_3B_UR50D": "ESM2-3B",
    "biohub/ESMC-300M": "ESMC-300M",
    "biohub/ESMC-600M": "ESMC-600M",
    "biohub/ESMC-6B": "ESMC-6B",
}


def _model_cache_label(model_name):
    return str(model_name).replace("/", "__")


def _safe_file_component(value):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_.-")
    return safe or "run"


def _n_transformer_layers(model_name):
    match = re.search(r"esm2_t(\d+)_", model_name)
    if match:
        return int(match.group(1))
    if model_name in MODEL_LAYER_COUNTS:
        return MODEL_LAYER_COUNTS[model_name]
    raise ValueError(f"No layer count registered for {model_name!r}.")


def _model_family(model_name):
    return "ESMC" if "ESMC" in model_name else "ESM2"


def _model_job_defaults(model_name, size_millions):
    is_esmc = "ESMC" in model_name
    if size_millions > 1000:
        return {
            "embedding_partition": "dept_gpu",
            "embedding_gres": "gpu:1",
            "embedding_constraint": "L40|A100",
            "embedding_mem": "64G",
            "embedding_time": "24:00:00",
            "torch_dtype": "bfloat16",
            "n_chunks": 40,
        }
    if is_esmc:
        return {
            "embedding_partition": "dept_gpu",
            "embedding_gres": "gpu:1",
            "embedding_constraint": "C8",
            "embedding_mem": "64G",
            "embedding_time": "12:00:00",
            "torch_dtype": "bfloat16",
            "n_chunks": 40,
        }
    return {
        "embedding_partition": "any_cpu",
        "embedding_gres": None,
        "embedding_constraint": None,
        "embedding_mem": "64G" if size_millions >= 650 else "48G",
        "embedding_time": "12:00:00" if size_millions >= 650 else "08:00:00",
        "torch_dtype": None,
        "n_chunks": 40,
    }

MODEL_SPECS = []
for model_name in AVAILABLE_MODEL_NAMES:
    size = MODEL_SIZE_MILLIONS[model_name]
    n_layers = _n_transformer_layers(model_name)
    spec = {
        "model": model_name,
        "model_short": MODEL_SHORT_NAMES.get(model_name, model_name),
        "model_cache_label": _model_cache_label(model_name),
        "family": _model_family(model_name),
        "size_millions": size,
        "n_transformer_layers": n_layers,
        "n_hidden_state_layers": n_layers + 1,
        "layers": list(range(n_layers + 1)),
        "default_include": size <= 1000,
        "requires_custom_gpu": size > 1000,
    }
    spec.update(_model_job_defaults(model_name, size))
    MODEL_SPECS.append(spec)

MODEL_SPECS_BY_MODEL = {spec["model"]: spec for spec in MODEL_SPECS}
ACTIVE_MODELS = [spec["model"] for spec in MODEL_SPECS if spec["default_include"]]
CUSTOM_GPU_MODELS = [spec["model"] for spec in MODEL_SPECS if spec["requires_custom_gpu"]]
INCLUDE_CUSTOM_GPU_MODELS_IN_ANALYSIS = False
ANALYSIS_MODELS = ACTIVE_MODELS + (CUSTOM_GPU_MODELS if INCLUDE_CUSTOM_GPU_MODELS_IN_ANALYSIS else [])

model_registry_df = pd.DataFrame([{k: v for k, v in spec.items() if k != "layers"} for spec in MODEL_SPECS])
model_registry_df.to_csv(TABLE_DIR / "BRCA1_plm_model_registry.csv", index=False)
display(model_registry_df)
print("Default submitted models:", ACTIVE_MODELS)
print("Large custom-GPU models kept available:", CUSTOM_GPU_MODELS)


## Process Counts And Scores

The processed BRCA1 sequence/count state is copied into per-model runners so every model shares the same input rows and mutation maps.


In [ ]:
def make_runner(model_name, save_dir=SEQUENCE_DIR):
    config = ESMDMSConfig(
        embedding_model=model_name,
        embedding_type=EMBEDDING_TYPE,
        local_or_disk="both",
        save_dir=str(save_dir),
        dataset_name=DATASET_NAME,
    )
    model_runner = esmDMS(input_data=BRCA1_INPUT, config=config)
    if "runner" in globals() and getattr(runner, "sequence_dataframe", None) is not None:
        model_runner.reference_sequence = runner.reference_sequence
        model_runner.sequence_dataframe = runner.sequence_dataframe
        model_runner.sequence_to_mutation_sites = runner.sequence_to_mutation_sites
        model_runner.sequence_to_protein_sequence = runner.sequence_to_protein_sequence
        model_runner.sequence_metadata = runner.sequence_metadata
        model_runner.reference_kind = runner.reference_kind
        model_runner.scores_dataframe = runner.scores_dataframe
    return model_runner

runner = make_runner(ACTIVE_MODELS[0])
runner.process_raw_data(drop_stop_codons=True)
runner.load_functional_scores()

processing_summary = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "reference_kind": runner.reference_kind,
    "count_rows": len(runner.sequence_dataframe),
    "mutation_keys_in_counts": runner.sequence_dataframe["SequenceIndex"].nunique(),
    "protein_sequence_keys": len(runner.sequence_to_protein_sequence),
    "replicates": runner.sequence_dataframe["Replicate"].nunique(),
    "generations": sorted(runner.sequence_dataframe["Generation"].unique()),
    "score_rows": len(runner.scores_dataframe),
    "skipped_counts": runner.sequence_metadata.attrs.get("skipped_counts", {}),
}])
processing_summary.to_csv(TABLE_DIR / "BRCA1_processing_summary.csv", index=False)
processing_summary


## Submit Embedding Jobs

One Slurm array is created per default sub-1B model. Array throttles are distributed so the total possible active embedding tasks is approximately `GLOBAL_MAX_ACTIVE_EMBEDDING_TASKS` across models. Set `SUBMIT_EMBEDDING_JOBS=True` to call `sbatch`.


In [ ]:
CREATE_EMBEDDING_JOBS = True
SUBMIT_EMBEDDING_JOBS = False
GLOBAL_MAX_ACTIVE_EMBEDDING_TASKS = 10
PYTHON_EXECUTABLE = "python3"
SCRATCH_ROOT = "/scr"
HF_HOME = None


def _active_slot_limits(models, total_slots):
    if not models:
        return {}
    if len(models) > total_slots:
        return {model: 1 for model in models}
    base = total_slots // len(models)
    extra = total_slots % len(models)
    return {model: base + (idx < extra) for idx, model in enumerate(models)}

embedding_active_limits = _active_slot_limits(ACTIVE_MODELS, GLOBAL_MAX_ACTIVE_EMBEDDING_TASKS)
embedding_job_rows = []

if CREATE_EMBEDDING_JOBS:
    for model_name in ACTIVE_MODELS:
        spec = MODEL_SPECS_BY_MODEL[model_name]
        model_runner = make_runner(model_name)
        job_dir = JOB_DIR / "embedding_batches" / spec["model_cache_label"]
        job = model_runner.create_embedding_batch_job(
            job_dir=job_dir,
            n_chunks=spec["n_chunks"],
            max_active_jobs=embedding_active_limits[model_name],
            job_name=_safe_file_component(f"brca1_{spec['model_short']}_embed")[:48],
            partition=spec["embedding_partition"],
            gres=spec["embedding_gres"],
            constraint=spec["embedding_constraint"],
            cpus_per_task=4,
            mem=spec["embedding_mem"],
            time=spec["embedding_time"],
            python_executable=PYTHON_EXECUTABLE,
            scratch_root=SCRATCH_ROOT,
            hf_home=HF_HOME,
            torch_dtype=spec["torch_dtype"],
            submit=SUBMIT_EMBEDDING_JOBS,
        )
        embedding_job_rows.append({
            "model": model_name,
            "model_short": spec["model_short"],
            "n_chunks": spec["n_chunks"],
            "max_active_jobs": embedding_active_limits[model_name],
            "partition": spec["embedding_partition"],
            "gres": spec["embedding_gres"],
            "constraint": spec["embedding_constraint"],
            "mem": spec["embedding_mem"],
            "time": spec["embedding_time"],
            "script_path": str(job["script_path"]),
            "payload_path": str(job["payload_path"]),
            "job_id": job["job_id"],
            "submitted": SUBMIT_EMBEDDING_JOBS,
        })

embedding_jobs_df = pd.DataFrame(embedding_job_rows)
embedding_jobs_df.to_csv(TABLE_DIR / "BRCA1_embedding_jobs.csv", index=False)
display(embedding_jobs_df)

custom_gpu_job_hints = pd.DataFrame([
    {
        "model": model_name,
        "model_short": MODEL_SPECS_BY_MODEL[model_name]["model_short"],
        "n_hidden_state_layers": MODEL_SPECS_BY_MODEL[model_name]["n_hidden_state_layers"],
        "recommended_partition": MODEL_SPECS_BY_MODEL[model_name]["embedding_partition"],
        "recommended_gres": MODEL_SPECS_BY_MODEL[model_name]["embedding_gres"],
        "recommended_constraint": MODEL_SPECS_BY_MODEL[model_name]["embedding_constraint"],
        "recommended_mem": MODEL_SPECS_BY_MODEL[model_name]["embedding_mem"],
        "recommended_time": MODEL_SPECS_BY_MODEL[model_name]["embedding_time"],
        "analysis_flag": "set INCLUDE_CUSTOM_GPU_MODELS_IN_ANALYSIS=True after outputs exist",
    }
    for model_name in CUSTOM_GPU_MODELS
])
custom_gpu_job_hints.to_csv(TABLE_DIR / "BRCA1_custom_gpu_model_hints.csv", index=False)
display(custom_gpu_job_hints)


## Merge Embedding Chunks

Run this after embedding arrays finish. You can either merge in the notebook or create one merge job per model.


In [ ]:
MERGE_EMBEDDING_OUTPUTS_LOCALLY = False
CREATE_EMBEDDING_MERGE_JOBS = True
SUBMIT_EMBEDDING_MERGE_JOBS = False

merge_rows = []
if MERGE_EMBEDDING_OUTPUTS_LOCALLY or CREATE_EMBEDDING_MERGE_JOBS:
    for model_name in ANALYSIS_MODELS:
        spec = MODEL_SPECS_BY_MODEL[model_name]
        model_runner = make_runner(model_name)
        job_dir = JOB_DIR / "embedding_batches" / spec["model_cache_label"]
        if MERGE_EMBEDDING_OUTPUTS_LOCALLY:
            model_runner.merge_embedding_batch_outputs(job_dir=job_dir, layer="all", n_chunks=spec["n_chunks"], save_layers=True)
        if CREATE_EMBEDDING_MERGE_JOBS:
            merge_job = model_runner.create_embedding_batch_merge_job(
                job_dir=job_dir,
                layer="all",
                n_chunks=spec["n_chunks"],
                save_layers=True,
                job_name=_safe_file_component(f"brca1_{spec['model_short']}_merge")[:48],
                partition="any_cpu",
                cpus_per_task=1,
                mem="16G",
                time="02:00:00",
                python_executable=PYTHON_EXECUTABLE,
                submit=SUBMIT_EMBEDDING_MERGE_JOBS,
            )
            merge_rows.append({
                "model": model_name,
                "model_short": spec["model_short"],
                "script_path": str(merge_job["script_path"]),
                "payload_path": str(merge_job["payload_path"]),
                "job_id": merge_job["job_id"],
                "submitted": SUBMIT_EMBEDDING_MERGE_JOBS,
            })

embedding_merge_jobs_df = pd.DataFrame(merge_rows)
embedding_merge_jobs_df.to_csv(TABLE_DIR / "BRCA1_embedding_merge_jobs.csv", index=False)
display(embedding_merge_jobs_df)


## Embedding Cache Status

This table is the gate for the SAE and raw-embedding analysis. Every included model/layer needs a `max_pool` cache.


In [ ]:
cache_rows = []
for model_name in ANALYSIS_MODELS:
    spec = MODEL_SPECS_BY_MODEL[model_name]
    model_runner = make_runner(model_name)
    for layer in spec["layers"]:
        path = model_runner._embedding_path(layer, EMBEDDING_TYPE)
        cache_rows.append({
            "model": model_name,
            "model_short": spec["model_short"],
            "layer": layer,
            "embedding_type": EMBEDDING_TYPE,
            "path": str(path),
            "exists": path.is_file(),
        })

embedding_cache_status_df = pd.DataFrame(cache_rows)
embedding_cache_status_df.to_csv(TABLE_DIR / "BRCA1_plm_layer_embedding_cache_status.csv", index=False)
missing_cache_count = int((~embedding_cache_status_df["exists"]).sum())
print(f"Missing {missing_cache_count} of {len(embedding_cache_status_df)} max_pool embedding caches.")
display(embedding_cache_status_df)


## Submit Fixed DeltaEmbSAE Model-Layer Jobs

This creates one Slurm array over `(model, layer)` tasks and throttles it with `%10` by default. Each task trains the fixed DeltaEmbSAE model and runs popDMS feature inference for that model/layer.


In [ ]:
CREATE_SAE_ARRAY_JOB = True
SUBMIT_SAE_ARRAY_JOB = False
GLOBAL_MAX_ACTIVE_SAE_TASKS = 10
SAE_ARRAY_PARTITION = "any_cpu"
SAE_ARRAY_CPUS_PER_TASK = 4
SAE_ARRAY_MEM = "48G"
SAE_ARRAY_TIME = "08:00:00"
FORCE_RECOMPUTE_SAE = False
SAE_TASK_ROOT = JOB_DIR / "fixed_sae_model_layer_array"
SAE_LOG_DIR = SAE_TASK_ROOT / "logs"
SAE_TASK_ROOT.mkdir(parents=True, exist_ok=True)
SAE_LOG_DIR.mkdir(parents=True, exist_ok=True)

sae_tasks = []
for model_name in ANALYSIS_MODELS:
    spec = MODEL_SPECS_BY_MODEL[model_name]
    for layer in spec["layers"]:
        sae_tasks.append({
            "model": model_name,
            "model_short": spec["model_short"],
            "layer": layer,
            "method": ABSTRACTION_METHOD,
            "embedding_type": EMBEDDING_TYPE,
            "params": dict(BEST_SAE_PARAMS),
            "run_label": BEST_SAE_PARAMS["run_label"],
            "sweep_dir": str(SAE_TASK_ROOT / spec["model_cache_label"] / f"Layer_{layer}"),
        })

payload_path = SAE_TASK_ROOT / "BRCA1_fixed_deltaembsae_layer_payload.pkl"
runner_path = SAE_TASK_ROOT / "run_fixed_deltaembsae_layer_task.py"
script_path = SAE_TASK_ROOT / "submit_fixed_deltaembsae_layer_array.sh"

if CREATE_SAE_ARRAY_JOB:
    payload = {
        "input_data": BRCA1_INPUT,
        "dataset_name": DATASET_NAME,
        "save_dir": str(SEQUENCE_DIR),
        "sequence_dataframe": runner.sequence_dataframe,
        "sequence_to_mutation_sites": runner.sequence_to_mutation_sites,
        "sequence_to_protein_sequence": runner.sequence_to_protein_sequence,
        "sequence_metadata": runner.sequence_metadata,
        "scores_dataframe": runner.scores_dataframe,
        "tasks": sae_tasks,
        "force_recompute": FORCE_RECOMPUTE_SAE,
    }
    with payload_path.open("wb") as handle:
        pickle.dump(payload, handle)

    runner_script = r'''
from pathlib import Path
import json
import os
import pickle
import sys

sys.path.insert(0, r"__REPO_ROOT__")

import pandas as pd
import popDMS  # noqa: F401
from esmDMS import ESMDMSConfig, esmDMS


def main(payload_path, task_idx):
    payload_path = Path(payload_path)
    with payload_path.open("rb") as handle:
        payload = pickle.load(handle)
    task = payload["tasks"][task_idx]
    layer = task["layer"]
    model_name = task["model"]
    embedding_type = task["embedding_type"]
    method = task["method"]
    sweep_dir = Path(task["sweep_dir"])
    output_root = sweep_dir / "runs"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    output_root.mkdir(parents=True, exist_ok=True)

    config = ESMDMSConfig(
        embedding_model=model_name,
        embedding_type=embedding_type,
        local_or_disk="both",
        save_dir=payload["save_dir"],
        dataset_name=payload["dataset_name"],
    )
    runner = esmDMS(payload["input_data"], config)
    runner.sequence_dataframe = payload["sequence_dataframe"]
    runner.sequence_to_mutation_sites = payload["sequence_to_mutation_sites"]
    runner.sequence_to_protein_sequence = payload["sequence_to_protein_sequence"]
    runner.sequence_metadata = payload["sequence_metadata"]
    runner.scores_dataframe = payload["scores_dataframe"]

    embedding_path = runner._embedding_path(layer, embedding_type)
    if not embedding_path.is_file():
        raise FileNotFoundError(
            "Missing %s embedding cache for model=%s layer=%s: %s" % (embedding_type, model_name, layer, embedding_path)
        )

    params = dict(task["params"])
    params["run_label"] = task["run_label"]
    single_payload = {
        "input_data": payload["input_data"],
        "config": {
            "embedding_model": model_name,
            "embedding_type": embedding_type,
            "embedding_method": None,
            "local_or_disk": "both",
            "save_dir": payload["save_dir"],
            "dataset_name": payload["dataset_name"],
        },
        "sequence_dataframe": payload["sequence_dataframe"],
        "sequence_to_mutation_sites": payload["sequence_to_mutation_sites"],
        "sequence_to_protein_sequence": payload["sequence_to_protein_sequence"],
        "sequence_metadata": payload["sequence_metadata"],
        "scores_dataframe": payload["scores_dataframe"],
        "layer": layer,
        "method": method,
        "embedding_type": embedding_type,
        "embedding_path": str(embedding_path),
        "output_root": str(output_root),
        "sweep_dir": str(sweep_dir),
        "configs": [{"run_label": task["run_label"], "params": params}],
        "gpus": 1,
        "max_parallel_runs": 1,
        "run_inference": True,
        "force_recompute": bool(payload.get("force_recompute", False)),
        "require_cuda": False,
    }

    result = esmDMS._run_sae_sweep_config_safe(single_payload, config_idx=0, gpu_idx=None, scratch_dir=os.environ.get("TMPDIR"))
    result.update({
        "task_idx": task_idx,
        "model": model_name,
        "model_short": task["model_short"],
        "layer_index": layer,
        "model_label": "%s Layer_%s %s" % (task["model_short"], layer, task["run_label"]),
    })
    pd.DataFrame([result]).to_csv(sweep_dir / "sae_sweep_results.csv", index=False)
    with (sweep_dir / "task_result.json").open("w") as handle:
        json.dump(result, handle, indent=2)
    if result.get("status") != "ok":
        raise RuntimeError("SAE task failed for %s layer %s: %s" % (model_name, layer, result.get("error")))


if __name__ == "__main__":
    main(Path(sys.argv[1]), int(sys.argv[2]))
'''.replace("__REPO_ROOT__", str(REPO_ROOT))
    runner_path.write_text(runner_script)
    runner_path.chmod(0o755)

    array_spec = f"0-{len(sae_tasks) - 1}%{GLOBAL_MAX_ACTIVE_SAE_TASKS}"
    slurm_script = f'''#!/bin/bash
#SBATCH --job-name=brca1_fixed_sae_layers
#SBATCH -p {SAE_ARRAY_PARTITION}
#SBATCH --cpus-per-task={SAE_ARRAY_CPUS_PER_TASK}
#SBATCH --time={SAE_ARRAY_TIME}
#SBATCH --mem={SAE_ARRAY_MEM}
#SBATCH --array={array_spec}
#SBATCH --output={SAE_TASK_ROOT}/logs/slurm-%A_%a.out
#SBATCH --error={SAE_TASK_ROOT}/logs/slurm-%A_%a.err

set -euo pipefail
cd {REPO_ROOT}
mkdir -p {SAE_TASK_ROOT}/logs

SCRDIR={SCRATCH_ROOT}/${{SLURM_JOB_ID}}_${{SLURM_ARRAY_TASK_ID}}_fixed_sae
mkdir -p "$SCRDIR"
export TMPDIR="$SCRDIR"
export MPLCONFIGDIR="$SCRDIR/mplconfig"
export OMP_NUM_THREADS={SAE_ARRAY_CPUS_PER_TASK}
export MKL_NUM_THREADS={SAE_ARRAY_CPUS_PER_TASK}
export NUMEXPR_NUM_THREADS={SAE_ARRAY_CPUS_PER_TASK}
mkdir -p "$MPLCONFIGDIR"

{PYTHON_EXECUTABLE} {runner_path} {payload_path} $SLURM_ARRAY_TASK_ID
'''
    script_path.write_text(slurm_script)
    script_path.chmod(0o755)

sae_array_job_id = ""
if SUBMIT_SAE_ARRAY_JOB:
    completed = subprocess.run(["sbatch", str(script_path)], check=True, capture_output=True, text=True)
    sae_array_job_id = completed.stdout.strip()

sae_task_df = pd.DataFrame(sae_tasks)
sae_task_df.to_csv(TABLE_DIR / "BRCA1_fixed_deltaembsae_layer_tasks.csv", index=False)
print(f"SAE tasks: {len(sae_tasks)}")
print(f"Array script: {script_path}")
print(f"Array spec: 0-{len(sae_tasks)-1}%{GLOBAL_MAX_ACTIVE_SAE_TASKS}")
print(f"Submitted: {SUBMIT_SAE_ARRAY_JOB} {sae_array_job_id}")
display(sae_task_df.head())



## Collect SAE Results And Plot Reconstruction/Sparsity

Run after the SAE array completes. The plot shows reconstruction R2 and activation sparsity by model and layer.


In [ ]:
def _as_existing_path(value):
    if value is None or pd.isna(value):
        return None
    path = Path(value)
    return path if path.is_file() else None


def _load_pickle(path):
    with Path(path).open("rb") as handle:
        return pickle.load(handle)


def _reconstruction_metrics(viz_path):
    viz_path = _as_existing_path(viz_path)
    if viz_path is None:
        return {
            "reconstruction_mse": np.nan,
            "reconstruction_r2": np.nan,
            "activation_sparsity": np.nan,
            "activation_density": np.nan,
            "n_active_features": np.nan,
            "active_feature_fraction": np.nan,
            "final_train_loss": np.nan,
            "final_test_loss": np.nan,
        }
    viz = _load_pickle(viz_path)
    X = np.asarray(viz["X_original"], dtype=float)
    X_recon = np.asarray(viz["X_reconstructed"], dtype=float)
    test_idx = np.asarray(viz.get("test_idx") or [], dtype=int)
    eval_idx = test_idx if len(test_idx) else np.arange(X.shape[0])
    X_eval = X[eval_idx]
    X_recon_eval = X_recon[eval_idx]
    residual = X_eval - X_recon_eval
    mse = float(np.mean(residual ** 2))
    sse = float(np.sum(residual ** 2))
    centered = X_eval - X_eval.mean(axis=0, keepdims=True)
    sst = float(np.sum(centered ** 2))
    r2 = 1.0 - (sse / sst) if sst > 0 else np.nan
    Z_all = np.asarray(viz["Z_all"])
    activation_density = float((Z_all > 0).mean())
    active_mask = np.asarray(viz["active_mask"], dtype=bool)
    train_losses = viz.get("train_losses") or []
    test_losses = viz.get("test_losses") or []
    return {
        "reconstruction_mse": mse,
        "reconstruction_r2": r2,
        "activation_sparsity": 1.0 - activation_density,
        "activation_density": activation_density,
        "n_active_features": int(active_mask.sum()),
        "active_feature_fraction": float(active_mask.mean()),
        "final_train_loss": float(train_losses[-1]) if train_losses else np.nan,
        "final_test_loss": float(test_losses[-1]) if test_losses else np.nan,
    }

result_paths = sorted(SAE_TASK_ROOT.glob("*/Layer_*/sae_sweep_results.csv"))
if not result_paths:
    print(f"No SAE result CSVs found under {SAE_TASK_ROOT}. Run the SAE array first.")
    sae_layer_metrics = pd.DataFrame()
else:
    result_frames = [pd.read_csv(path) for path in result_paths]
    sae_layer_metrics = pd.concat(result_frames, ignore_index=True, sort=False)
    if "layer_index" not in sae_layer_metrics.columns and "layer" in sae_layer_metrics.columns:
        sae_layer_metrics["layer_index"] = sae_layer_metrics["layer"].astype(str).str.replace("Layer_", "", regex=False).astype(int)
    for column in ["n_features", "k", "elapsed_seconds", "feature_count", "feature_dim"]:
        if column in sae_layer_metrics.columns:
            sae_layer_metrics[column] = pd.to_numeric(sae_layer_metrics[column], errors="coerce")
    metric_rows = []
    for _, row in sae_layer_metrics.iterrows():
        metric_rows.append(_reconstruction_metrics(row.get("viz_path")))
    sae_layer_metrics = pd.concat([sae_layer_metrics.reset_index(drop=True), pd.DataFrame(metric_rows)], axis=1)
    sae_layer_metrics.to_csv(TABLE_DIR / "BRCA1_fixed_deltaembsae_layer_metrics.csv", index=False)

    plot_df = sae_layer_metrics[sae_layer_metrics["status"].eq("ok")].copy()
    if not plot_df.empty:
        fig, axes = plt.subplots(2, 1, figsize=(11.0, 8.0), sharex=True)
        sns.lineplot(data=plot_df, x="layer_index", y="reconstruction_r2", hue="model_short", marker="o", linewidth=1.5, ax=axes[0])
        axes[0].set_ylabel("SAE reconstruction R2")
        axes[0].set_xlabel("")
        axes[0].legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        sns.lineplot(data=plot_df, x="layer_index", y="activation_sparsity", hue="model_short", marker="o", linewidth=1.5, ax=axes[1], legend=False)
        axes[1].set_ylabel("Activation sparsity")
        axes[1].set_xlabel("Layer")
        fig.suptitle("Fixed DeltaEmbSAE reconstruction and sparsity across PLM layers")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "BRCA1_fixed_deltaembsae_reconstruction_sparsity_by_model_layer.png", dpi=FIGURE_DPI)
        plt.show()

display(sae_layer_metrics)


## ClinVar And Fitness Helpers

These helpers use the cached ClinVar annotations with review-star filters. Pathogenic is treated as the positive class and `-fitness` as the classifier score.


In [ ]:
PATHOGENICITY_LABELS = {"benign", "pathogenic"}


def _clinvar_annotation_table():
    if not CLINVAR_ANNOTATION_CACHE_PATH.is_file():
        raise FileNotFoundError(
            f"Missing cached ClinVar annotation table: {CLINVAR_ANNOTATION_CACHE_PATH}. "
            "Run the annotation section of real_data_esm_analysis_cpu_sae.ipynb first."
        )
    annotations = pd.read_csv(CLINVAR_ANNOTATION_CACHE_PATH)
    annotations["SequenceIndex"] = annotations[BRCA1_INPUT.primary_key].astype(str)
    annotations["stars"] = pd.to_numeric(annotations.get("stars", 0), errors="coerce").fillna(0)
    return annotations


def _clinvar_binary_annotation_map(min_review_stars=0):
    annotations = _clinvar_annotation_table()
    keep = annotations["annotation"].isin(PATHOGENICITY_LABELS)
    if min_review_stars > 0:
        keep = keep & annotations["stars"].ge(min_review_stars)
    annotations = annotations[keep].copy()
    return dict(zip(annotations["SequenceIndex"], annotations["annotation"]))


def _classification_metrics_for_fitness(fitness_df):
    binary_df = fitness_df[fitness_df["annotation"].isin(PATHOGENICITY_LABELS)].dropna(subset=["fitness"]).copy()
    if binary_df.empty:
        return {"auc": np.nan, "n_benign": 0, "n_pathogenic": 0, "n_variants": 0}
    labels = binary_df["annotation"].eq("pathogenic").to_numpy()
    scores = -pd.to_numeric(binary_df["fitness"], errors="coerce").to_numpy(dtype=float)
    finite = np.isfinite(scores)
    labels = labels[finite]
    scores = scores[finite]
    n_pos = int(labels.sum())
    n_neg = int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        auc = np.nan
    else:
        ranks = pd.Series(scores).rank(method="average").to_numpy()
        auc = float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
    return {"auc": auc, "n_benign": int(n_neg), "n_pathogenic": int(n_pos), "n_variants": int(len(labels))}


def _spearman_for_fitness_dataframe(fitness_df, score_col="score"):
    if runner.scores_dataframe is None:
        runner.load_functional_scores()
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    score_df["SequenceIndex"] = score_df["SequenceIndex"].astype(str)
    comparison_df = fitness_df.merge(score_df, on="SequenceIndex", how="inner")
    fitness = pd.to_numeric(comparison_df["fitness"], errors="coerce").to_numpy(dtype=float)
    scores = pd.to_numeric(comparison_df[score_col], errors="coerce").to_numpy(dtype=float)
    finite = np.isfinite(fitness) & np.isfinite(scores)
    if finite.sum() < 3:
        return np.nan, int(finite.sum())
    rho = spearmanr(fitness[finite], scores[finite]).statistic
    return float(rho), int(finite.sum())


def _fitness_for_feature_mapping(seq_to_features, inference_result, annotation_map, norm_scheme=NORM_SCHEME):
    _, seq_to_features = esmDMS._drop_missing_features(None, seq_to_features, "Fitness scoring")
    esmDMS._require_vector_features(seq_to_features, "Fitness scoring")
    seq_ids = list(seq_to_features)
    features = np.asarray([seq_to_features[seq_id] for seq_id in seq_ids], dtype=float)
    if norm_scheme is not None and norm_scheme != "none":
        features = esmDMS._normalize_features(features, norm_scheme)
    if getattr(inference_result, "s_joint", None) is not None:
        fitness = 1.0 + features @ inference_result.s_joint
    else:
        fitness = 1.0 + np.asarray([features @ inference_result.s[rep_idx] for rep_idx in range(inference_result.s.shape[0])]).mean(axis=0)
    return pd.DataFrame({
        "SequenceIndex": [str(seq_id) for seq_id in seq_ids],
        "fitness": fitness,
        "annotation": [annotation_map.get(str(seq_id)) for seq_id in seq_ids],
    })


def _metrics_row_from_fitness(
    fitness_df,
    annotation_map,
    model_label,
    model_group,
    benchmark,
    method,
    embedding_type,
    model="",
    model_short="",
    layer=np.nan,
    n_features=np.nan,
    k=np.nan,
    feature_path="",
    inference_path="",
    plot_group=None,
):
    if "annotation" not in fitness_df.columns:
        fitness_df = fitness_df.copy()
        fitness_df["annotation"] = fitness_df["SequenceIndex"].astype(str).map(annotation_map)
    spearman_rho, n_score_sequences = _spearman_for_fitness_dataframe(fitness_df)
    auc_metrics = _classification_metrics_for_fitness(fitness_df)
    return {
        "dataset": DATASET_NAME,
        "model": model,
        "model_short": model_short,
        "layer": layer,
        "model_label": model_label,
        "model_group": model_group,
        "benchmark": benchmark,
        "method": method,
        "embedding_type": embedding_type,
        "n_features": n_features,
        "k": k,
        "spearman_rho": spearman_rho,
        "n_score_sequences": n_score_sequences,
        "feature_path": str(feature_path) if feature_path else "",
        "inference_path": str(inference_path) if inference_path else "",
        "method_pool": f"{method} / {embedding_type}",
        "plot_group": plot_group or model_group,
        **auc_metrics,
    }


## Build Benchmark Rows

Raw embedding benchmarks are restricted to `max_pool` and are evaluated by model and layer. Fixed DeltaEmbSAE rows are loaded from the SAE array outputs.


In [ ]:
RUN_RAW_MAX_POOL_BENCHMARKS = True
RUN_FIXED_SAE_BENCHMARKS = True
RUN_BASELINE_BENCHMARKS = True


def _functional_score_fitness_dataframe(annotation_map, score_col="score"):
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    score_df["SequenceIndex"] = score_df["SequenceIndex"].astype(str)
    score_df["fitness"] = pd.to_numeric(score_df[score_col], errors="coerce")
    score_df["annotation"] = score_df["SequenceIndex"].map(annotation_map)
    return score_df[["SequenceIndex", "fitness", "annotation"]]


def _enrichment_ratio_fitness_dataframe(annotation_map=None, pseudocount=0.5):
    counts_df = pd.read_csv(BRCA1_INPUT.mavedb_csv_path).copy()
    primary_key = BRCA1_INPUT.primary_key
    counts_df["SequenceIndex"] = counts_df[primary_key].astype(str)
    day_count_cols = []
    for column in counts_df.columns:
        match = re.fullmatch(r"count_day(?P<day>\d+)_rep(?P<rep>\d+)", str(column))
        if match:
            day_count_cols.append((int(match.group("day")), int(match.group("rep")), column))
    if not day_count_cols:
        raise ValueError("No count_day<day>_rep<rep> columns were found for enrichment ratio baseline.")
    final_day = max(day for day, _, _ in day_count_cols)
    final_cols = [(rep, column) for day, rep, column in day_count_cols if day == final_day]
    day0_cols = {rep: column for day, rep, column in day_count_cols if day == 0}
    n_rows = len(counts_df)
    replicate_scores = []
    for rep, final_col in sorted(final_cols):
        initial_col = day0_cols.get(rep, "count_library")
        initial_counts = pd.to_numeric(counts_df[initial_col], errors="coerce").fillna(0.0)
        final_counts = pd.to_numeric(counts_df[final_col], errors="coerce").fillna(0.0)
        initial_freq = (initial_counts + pseudocount) / (initial_counts.sum() + pseudocount * n_rows)
        final_freq = (final_counts + pseudocount) / (final_counts.sum() + pseudocount * n_rows)
        replicate_scores.append(np.log2(final_freq / initial_freq))
    enrichment = pd.concat(replicate_scores, axis=1).mean(axis=1)
    fitness_df = pd.DataFrame({"SequenceIndex": counts_df["SequenceIndex"], "fitness": enrichment})
    if annotation_map is not None:
        fitness_df["annotation"] = fitness_df["SequenceIndex"].map(annotation_map)
    fitness_df.to_csv(ENRICHMENT_RATIO_FITNESS_PATH, index=False)
    return fitness_df


def _regular_popdms_fitness_dataframe(annotation_map):
    if not EXACT_POPDMS_FITNESS_PATH.is_file():
        raise FileNotFoundError(f"Missing exact popDMS fitness table: {EXACT_POPDMS_FITNESS_PATH}")
    fitness_df = pd.read_csv(EXACT_POPDMS_FITNESS_PATH)
    fitness_df["SequenceIndex"] = fitness_df["SequenceIndex"].astype(str)
    fitness_df["annotation"] = fitness_df["SequenceIndex"].map(annotation_map)
    return fitness_df[["SequenceIndex", "fitness", "annotation"]]


def _raw_max_pool_fitness_dataframe(model_name, layer, annotation_map):
    model_runner = make_runner(model_name)
    model_runner.run_feature_inference(
        layer=layer,
        abstraction_method="none",
        abstraction_params={"norm_scheme": NORM_SCHEME},
        embedding_type=EMBEDDING_TYPE,
    )
    return model_runner.fitness_dataframe(
        layer=layer,
        abstraction_method="none",
        abstraction_params={"norm_scheme": NORM_SCHEME},
        embedding_type=EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
    ).assign(annotation=lambda df: df["SequenceIndex"].astype(str).map(annotation_map))


def _fixed_sae_fitness_dataframe(row, annotation_map):
    feature_path = _as_existing_path(row.get("feature_path"))
    inference_path = _as_existing_path(row.get("inference_path"))
    if feature_path is None or inference_path is None:
        raise FileNotFoundError(f"Missing fixed SAE artifacts for {row.get('model_label', 'row')}")
    return _fitness_for_feature_mapping(_load_pickle(feature_path), _load_pickle(inference_path), annotation_map)


def _build_spearman_auc_metrics(annotation_map):
    rows = []
    if RUN_BASELINE_BENCHMARKS:
        dms_fitness_df = _functional_score_fitness_dataframe(annotation_map)
        rows.append(_metrics_row_from_fitness(
            dms_fitness_df, annotation_map, "DMS functional score", "benchmark", "DMS functional score", "DMS functional score", "score", plot_group="Raw max-pool baseline"
        ))
        enrichment_df = _enrichment_ratio_fitness_dataframe(annotation_map)
        rows.append(_metrics_row_from_fitness(
            enrichment_df, annotation_map, "Enrichment ratio", "Enrichment ratio", "Enrichment ratio", "enrichment ratio", "day11 / day0 counts", plot_group="Enrichment ratio"
        ))
        try:
            regular_df = _regular_popdms_fitness_dataframe(annotation_map)
            rows.append(_metrics_row_from_fitness(
                regular_df, annotation_map, "Regular popDMS", "Regular popDMS", "Regular popDMS", "regular popDMS", "amino-acid haplotype frequencies", inference_path=EXACT_POPDMS_SELECTION_PATH, plot_group="Regular popDMS"
            ))
        except FileNotFoundError as exc:
            print(f"Skipping regular popDMS baseline: {exc}")

    if RUN_RAW_MAX_POOL_BENCHMARKS:
        for model_name in ANALYSIS_MODELS:
            spec = MODEL_SPECS_BY_MODEL[model_name]
            model_runner = make_runner(model_name)
            for layer in spec["layers"]:
                embedding_path = model_runner._embedding_path(layer, EMBEDDING_TYPE)
                if not embedding_path.is_file():
                    continue
                fitness_df = _raw_max_pool_fitness_dataframe(model_name, layer, annotation_map)
                rows.append(_metrics_row_from_fitness(
                    fitness_df,
                    annotation_map,
                    model_label=f"{spec['model_short']} Layer_{layer} raw max_pool",
                    model_group="benchmark",
                    benchmark="Raw embeddings",
                    method="raw ESM",
                    embedding_type=EMBEDDING_TYPE,
                    model=model_name,
                    model_short=spec["model_short"],
                    layer=layer,
                    feature_path=embedding_path,
                    inference_path=model_runner._inference_path("none", layer, NORM_SCHEME, EMBEDDING_TYPE),
                    plot_group="Raw max-pool baseline",
                ))

    if RUN_FIXED_SAE_BENCHMARKS:
        metrics_path = TABLE_DIR / "BRCA1_fixed_deltaembsae_layer_metrics.csv"
        if "sae_layer_metrics" in globals() and isinstance(sae_layer_metrics, pd.DataFrame) and not sae_layer_metrics.empty:
            fixed_metrics_df = sae_layer_metrics.copy()
        elif metrics_path.is_file():
            fixed_metrics_df = pd.read_csv(metrics_path)
        else:
            fixed_metrics_df = pd.DataFrame()
        if fixed_metrics_df.empty:
            print("No fixed SAE metrics found; run and collect the SAE array before plotting fixed SAE points.")
        else:
            fixed_metrics_df = fixed_metrics_df[fixed_metrics_df["status"].eq("ok")].copy()
            for _, row in fixed_metrics_df.iterrows():
                fitness_df = _fixed_sae_fitness_dataframe(row, annotation_map)
                rows.append(_metrics_row_from_fitness(
                    fitness_df,
                    annotation_map,
                    model_label=row.get("model_label", f"{row.get('model_short')} Layer_{row.get('layer_index')} fixed DeltaEmbSAE"),
                    model_group="Fixed DeltaEmbSAE",
                    benchmark="Fixed DeltaEmbSAE",
                    method="DeltaEmbSAE",
                    embedding_type=EMBEDDING_TYPE,
                    model=row.get("model", ""),
                    model_short=row.get("model_short", ""),
                    layer=row.get("layer_index", np.nan),
                    n_features=row.get("n_features", BEST_SAE_PARAMS["n_features"]),
                    k=row.get("k", BEST_SAE_PARAMS["k"]),
                    feature_path=row.get("feature_path", ""),
                    inference_path=row.get("inference_path", ""),
                    plot_group="Fixed DeltaEmbSAE",
                ))
    return pd.DataFrame(rows)

annotation_map = _clinvar_binary_annotation_map(min_review_stars=0)
spearman_auc_df = _build_spearman_auc_metrics(annotation_map)
spearman_auc_df.to_csv(SPEARMAN_AUC_TABLE_PATH, index=False)
display(spearman_auc_df.sort_values(["plot_group", "auc", "spearman_rho"], ascending=[True, False, False]))


## Functional Score Agreement Vs ClinVar Classification

This reproduces the MaveDB Spearman-vs-ClinVar AUC plot and repeats it for ClinVar review-star thresholds.


In [ ]:
def _fitness_dataframe_for_metrics_row(row, annotation_map):
    benchmark = row.get("benchmark", "")
    if benchmark == "DMS functional score":
        return _functional_score_fitness_dataframe(annotation_map, score_col="score")
    if benchmark == "Enrichment ratio":
        return _enrichment_ratio_fitness_dataframe(annotation_map)
    if benchmark == "Regular popDMS":
        return _regular_popdms_fitness_dataframe(annotation_map)
    if benchmark == "Raw embeddings":
        return _raw_max_pool_fitness_dataframe(row["model"], int(row["layer"]), annotation_map)
    if benchmark == "Fixed DeltaEmbSAE":
        return _fixed_sae_fitness_dataframe(row, annotation_map)
    raise ValueError(f"Unsupported benchmark row: {benchmark}")


def _recompute_auc_for_annotation_map(base_metrics_df, annotation_map):
    rows = []
    for _, row in base_metrics_df.iterrows():
        try:
            fitness_df = _fitness_dataframe_for_metrics_row(row, annotation_map)
        except Exception as exc:
            print(f"Skipping {row.get('model_label', 'model')} for star-filtered AUC: {exc}")
            continue
        row_dict = row.to_dict()
        row_dict.update(_classification_metrics_for_fitness(fitness_df))
        rows.append(row_dict)
    return pd.DataFrame(rows)


def _plot_mavedb_spearman_vs_clinvar_auc(comparison_df, output_path, functional_score_auc=np.nan, title_suffix=""):
    plot_df = comparison_df[np.isfinite(comparison_df["spearman_rho"]) & np.isfinite(comparison_df["auc"])].copy()
    if plot_df.empty:
        print("No finite Spearman/AUC pairs were available to plot.")
        return None
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    fixed_df = plot_df[plot_df["plot_group"].eq("Fixed DeltaEmbSAE")].copy()
    if not fixed_df.empty:
        sns.scatterplot(data=fixed_df, x="spearman_rho", y="auc", hue="model_short", s=58, alpha=0.62, edgecolor="white", linewidth=0.45, ax=ax)
        best_auc_row = fixed_df.loc[fixed_df["auc"].idxmax()]
        best_rho_row = fixed_df.loc[fixed_df["spearman_rho"].idxmax()]
        for label, row in [("Best fixed SAE AUC", best_auc_row), ("Best fixed SAE rho", best_rho_row)]:
            ax.annotate(label, (row["spearman_rho"], row["auc"]), xytext=(6, -9), textcoords="offset points", fontsize=7, ha="left", va="top", color="0.18")

    marker_specs = {
        "Raw embeddings": {"marker": "D", "size": 90, "color": "#f58518", "label": "Raw max_pool embeddings"},
        "Regular popDMS": {"marker": "*", "size": 230, "color": "#111111", "label": "Regular popDMS"},
        "Enrichment ratio": {"marker": "v", "size": 150, "color": "#009e73", "label": "Enrichment ratio"},
    }
    baseline_df = plot_df[~plot_df["plot_group"].eq("Fixed DeltaEmbSAE") & ~plot_df["benchmark"].eq("DMS functional score")].copy()
    for benchmark, group_df in baseline_df.groupby("benchmark", sort=False):
        spec = marker_specs.get(benchmark, {"marker": "o", "size": 90, "color": "0.5", "label": benchmark})
        ax.scatter(group_df["spearman_rho"], group_df["auc"], s=spec["size"], marker=spec["marker"], color=spec["color"], alpha=0.95, edgecolors="white", linewidths=0.8, label=spec["label"], zorder=5)

    if np.isfinite(functional_score_auc):
        ax.axhline(functional_score_auc, color="0.20", linestyle="-.", linewidth=1.15, label=f"MaveDB functional scores AUC={functional_score_auc:.3f}", zorder=1)
    ax.axhline(0.5, color="0.35", linestyle="--", linewidth=1.0, zorder=0)
    ax.axvline(0.0, color="0.72", linestyle=":", linewidth=1.0, zorder=0)
    x_values = plot_df["spearman_rho"].to_numpy(dtype=float)
    y_values = plot_df["auc"].to_numpy(dtype=float)
    if np.isfinite(functional_score_auc):
        y_values = np.append(y_values, functional_score_auc)
    x_pad = max(0.025, 0.08 * (np.nanmax(x_values) - np.nanmin(x_values)))
    y_pad = max(0.025, 0.08 * (np.nanmax(y_values) - np.nanmin(y_values)))
    ax.set_xlim(max(-1.0, np.nanmin(x_values) - x_pad), min(1.0, np.nanmax(x_values) + x_pad))
    ax.set_ylim(max(0.0, min(np.nanmin(y_values) - y_pad, 0.47)), min(1.0, max(np.nanmax(y_values) + y_pad, 0.53)))
    ax.set_xlabel("MaveDB functional-score Spearman rho")
    ax.set_ylabel("ClinVar pathogenic-vs-benign AUC")
    title = "Functional score agreement vs ClinVar classification"
    if title_suffix:
        title = f"{title}\n{title_suffix}"
    ax.set_title(title)
    ax.grid(axis="both", color="0.90", linewidth=0.8)
    handles, labels = ax.get_legend_handles_labels()
    seen = set()
    unique = []
    for handle, label in zip(handles, labels):
        if label not in seen:
            seen.add(label)
            unique.append((handle, label))
    if unique:
        ax.legend([handle for handle, _ in unique], [label for _, label in unique], bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, borderaxespad=0)
    fig.tight_layout()
    fig.savefig(output_path, dpi=FIGURE_DPI)
    return fig


def _clinvar_agreement_summary_table(metrics_df, functional_score_auc_metrics, min_review_stars=None):
    def best_auc(mask):
        values = pd.to_numeric(metrics_df.loc[mask, "auc"], errors="coerce")
        values = values[np.isfinite(values)]
        return np.nan if values.empty else float(values.max())
    summary = pd.DataFrame([{
        "review_filter": "all binary annotations" if min_review_stars is None else f">= {min_review_stars} star(s)",
        "best_raw_max_pool_auc": best_auc(metrics_df["benchmark"].eq("Raw embeddings")),
        "best_fixed_sae_auc": best_auc(metrics_df["plot_group"].eq("Fixed DeltaEmbSAE")),
        "mavedb_functional_auc": functional_score_auc_metrics.get("auc", np.nan),
        "popdms_auc": best_auc(metrics_df["benchmark"].eq("Regular popDMS")),
        "n_benign_annotations": int(functional_score_auc_metrics.get("n_benign", 0) or 0),
        "n_pathogenic_annotations": int(functional_score_auc_metrics.get("n_pathogenic", 0) or 0),
    }])
    for column in ["best_raw_max_pool_auc", "best_fixed_sae_auc", "mavedb_functional_auc", "popdms_auc"]:
        summary[column] = pd.to_numeric(summary[column], errors="coerce").round(3)
    return summary

annotation_map = _clinvar_binary_annotation_map(min_review_stars=0)
functional_score_auc_metrics = _classification_metrics_for_fitness(_functional_score_fitness_dataframe(annotation_map))
if "spearman_auc_df" not in globals() or spearman_auc_df.empty:
    spearman_auc_df = _build_spearman_auc_metrics(annotation_map)
spearman_auc_df.to_csv(SPEARMAN_AUC_TABLE_PATH, index=False)

fig = _plot_mavedb_spearman_vs_clinvar_auc(spearman_auc_df, SPEARMAN_AUC_FIGURE_PATH, functional_score_auc=functional_score_auc_metrics["auc"])
if fig is not None:
    plt.show()
    display(_clinvar_agreement_summary_table(spearman_auc_df, functional_score_auc_metrics, min_review_stars=None))

star_threshold_frames = []
for min_review_stars in CLINVAR_REVIEW_STAR_THRESHOLDS:
    threshold_annotation_map = _clinvar_binary_annotation_map(min_review_stars)
    threshold_metrics_df = _recompute_auc_for_annotation_map(spearman_auc_df, threshold_annotation_map)
    if threshold_metrics_df.empty:
        print(f"No models could be plotted for ClinVar annotations >= {min_review_stars} star(s).")
        continue
    threshold_metrics_df["min_review_stars"] = min_review_stars
    threshold_metrics_df.to_csv(TABLE_DIR / SPEARMAN_AUC_STAR_TABLE_TEMPLATE.format(stars=min_review_stars), index=False)
    threshold_functional_score_auc = _classification_metrics_for_fitness(_functional_score_fitness_dataframe(threshold_annotation_map))
    threshold_fig = _plot_mavedb_spearman_vs_clinvar_auc(
        threshold_metrics_df,
        FIGURE_DIR / SPEARMAN_AUC_STAR_FIGURE_TEMPLATE.format(stars=min_review_stars),
        functional_score_auc=threshold_functional_score_auc["auc"],
        title_suffix=f"ClinVar annotations >= {min_review_stars} review star(s)",
    )
    if threshold_fig is not None:
        plt.show()
        display(_clinvar_agreement_summary_table(threshold_metrics_df, threshold_functional_score_auc, min_review_stars=min_review_stars))
    star_threshold_frames.append(threshold_metrics_df)

if star_threshold_frames:
    spearman_auc_by_review_stars_df = pd.concat(star_threshold_frames, ignore_index=True, sort=False)
    spearman_auc_by_review_stars_df.to_csv(SPEARMAN_AUC_STAR_COMBINED_TABLE_PATH, index=False)
    display(spearman_auc_by_review_stars_df.sort_values(["min_review_stars", "plot_group", "auc", "spearman_rho"], ascending=[True, True, False, False]))
